In [1]:
EVENTS = [
    {"ticket_id": "T1", "ts": "2026-02-10T09:12:11", "event": "open",       "agent": None,    "customer": "C001", "meta": {"channel": "email"}},
    {"ticket_id": "T1", "ts": "2026-02-10T09:14:03", "event": "assign",     "agent": "a12",   "customer": "C001", "meta": {"priority": "P2"}},
    {"ticket_id": "T1", "ts": "2026-02-10T10:05:00", "event": "message",    "agent": "a12",   "customer": "C001", "meta": {"body_len": "418"}},  # str
    {"ticket_id": "T1", "ts": "2026-02-10T11:20:00", "event": "resolve",    "agent": "a12",   "customer": "C001", "meta": {"resolution": "reboot"}},

    {"ticket_id": "T2", "ts": "2026-02-10T10:00:00", "event": "open",       "agent": None,    "customer": "C002", "meta": {"channel": "web"}},
    {"ticket_id": "T2", "ts": "2026-02-10T10:10:00", "event": "message",    "agent": None,    "customer": "C002", "meta": {"body_len": 120}},
    {"ticket_id": "T2", "ts": "2026-02-10T12:00:00", "event": "assign",     "agent": "a07",   "customer": "C002", "meta": {"priority": "P1"}},
    {"ticket_id": "T2", "ts": "2026-02-11T09:00:00", "event": "resolve",    "agent": "a07",   "customer": "C002", "meta": {"resolution": "patch"}},

    {"ticket_id": "T3", "ts": "2026-02-11T08:00:00", "event": "open",       "agent": None,    "customer": "C003", "meta": {"channel": "email"}},
    {"ticket_id": "T3", "ts": "2026-02-11T08:05:00", "event": "assign",     "agent": "a12",   "customer": "C003", "meta": {"priority": "P3"}},
    {"ticket_id": "T3", "ts": "2026-02-11T08:10:00", "event": "message",    "agent": "a12",   "customer": "C003", "meta": {"body_len": None}},     # None
    {"ticket_id": "T3", "ts": "2026-02-12T18:30:00", "event": "reopen",     "agent": None,    "customer": "C003", "meta": {}},
    {"ticket_id": "T3", "ts": "2026-02-12T19:00:00", "event": "assign",     "agent": "a12",   "customer": "C003", "meta": {"priority": "P2"}},
    {"ticket_id": "T3", "ts": "2026-02-12T21:15:00", "event": "resolve",    "agent": "a12",   "customer": "C003", "meta": {"resolution": "replace"}},

    {"ticket_id": "T4", "ts": "2026-02-13T07:00:00", "event": "assign",     "agent": "a07",   "customer": "C004", "meta": {"priority": "P2"}},      # assign before open
    {"ticket_id": "T4", "ts": "2026-02-13T07:02:00", "event": "open",       "agent": None,    "customer": "C004", "meta": {"channel": "phone"}},
    {"ticket_id": "T4", "ts": "2026-02-13T07:05:00", "event": "resolve",    "agent": "a07",   "customer": "C004", "meta": {"resolution": "advise"}},

    {"ticket_id": None, "ts": "2026-02-13T08:00:00", "event": "open",       "agent": None,    "customer": "C005", "meta": {"channel": "web"}},       # invalid id
    {"ticket_id": "T5", "ts": "BAD_TS",              "event": "open",       "agent": None,    "customer": "C006", "meta": {"channel": "web"}},       # bad timestamp
]

In [16]:
from datetime import datetime
from typing import Any, Dict, List, Tuple, Optional


def parse_dt_strict(ts: Any) -> Optional[datetime]:
    if not isinstance(ts, str):
        return None
    try:
        return datetime.strptime(ts, "%Y-%m-%dT%H:%M:%S")
    except ValueError:
        return None


def normalize_body_len(meta: Any) -> Optional[int]:
    # Returns None if body_len key is absent; otherwise returns normalized int
    if not isinstance(meta, dict):
        return None
    if "body_len" not in meta:
        return None

    v = meta.get("body_len")
    if v is None:
        return 0
    if isinstance(v, int):
        return v
    if isinstance(v, str):
        try:
            return int(v)
        except ValueError:
            return 0
    return 0


def normalize_events(events: List[Dict[str, Any]]) -> Tuple[List[Dict[str, Any]], List[Dict[str, Any]]]:
    clean: List[Dict[str, Any]] = []
    rejected: List[Dict[str, Any]] = []

    for e in events:
        ticket_id = e.get("ticket_id")
        ts = e.get("ts")
        ev_name = e.get("event")

        if not isinstance(ticket_id, str) or ticket_id.strip() == "":
            r = dict(e)
            r["reason"] = "invalid_ticket_id"
            rejected.append(r)
            continue

        if not isinstance(ev_name, str) or ev_name.strip() == "":
            r = dict(e)
            r["reason"] = "invalid_event"
            rejected.append(r)
            continue

        dt = parse_dt_strict(ts)
        if dt is None:
            r = dict(e)
            r["reason"] = "invalid_ts"
            rejected.append(r)
            continue

        ne = dict(e)
        ne["dt"] = dt

        meta = ne.get("meta")
        body_len = normalize_body_len(meta)
        if body_len is not None and isinstance(meta, dict):
            new_meta = dict(meta)
            new_meta["body_len"] = body_len
            ne["meta"] = new_meta

        clean.append(ne)

    clean.sort(key=lambda x: (x["ticket_id"], x["dt"]))
    return clean, rejected

In [18]:
clean, rejected = normalize_events(EVENTS)

assert any(r.get("ticket_id") is None and r["reason"] == "invalid_ticket_id" for r in rejected)
assert any(r.get("ticket_id") == "T5" and r["reason"] == "invalid_ts" for r in rejected)

t1_msgs = [e for e in clean if e["ticket_id"] == "T1" and e["event"] == "message"]
assert len(t1_msgs) == 1 and t1_msgs[0]["meta"]["body_len"] == 418

t3_msgs = [e for e in clean if e["ticket_id"] == "T3" and e["event"] == "message"]
assert len(t3_msgs) == 1 and t3_msgs[0]["meta"]["body_len"] == 0

# no mutation: original still has "418" as string
assert EVENTS[2]["meta"]["body_len"] == "418"